# Step 0: Slicing (evangeline, patient 29, tumour)

Turns the 3D scan into 2D slices + masks. **Already done:** the slices are in `data/slices/evangeline/`.
Run this only if you need to make them again (needs `volume-29.nii.gz` and `segmentation-29.nii.gz` in `data/raw_volumes/`).

In [1]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import cv2

PATIENT = 29          # this member's patient
NAME = "evangeline"      # this member's slices folder
RAW = "../data/raw_volumes"
OUT = f"../data/slices/{NAME}"

In [2]:
vol = nib.load(f"{RAW}/volume-{PATIENT}.nii.gz").get_fdata()        # CT, values in HU
seg = nib.load(f"{RAW}/segmentation-{PATIENT}.nii.gz").get_fdata()  # 0 bg, 1 liver, 2 tumour
print("shape:", vol.shape, seg.shape)
print("HU range:", vol.min(), "to", vol.max())
print("labels in mask:", np.unique(seg))

shape: (512, 512, 172) (512, 512, 172)
HU range: -1024.0 to 1023.0


labels in mask: [0. 1. 2.]


In [3]:
tumour_slices = [i for i in range(seg.shape[2]) if np.any(seg[:, :, i] == 2)]
print(len(tumour_slices), "tumour slices:", tumour_slices)

22 tumour slices: [56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]


In [4]:
chosen = tumour_slices          # all tumour slices (they also contain liver)
os.makedirs(OUT, exist_ok=True)
for i in chosen:
    ct = np.rot90(vol[:, :, i])
    s = np.rot90(seg[:, :, i])
    base = f"{OUT}/p{PATIENT}_s{i}"
    np.save(base + "_ct.npy", ct.astype(np.float32))
    cv2.imwrite(base + "_liver.png", ((s >= 1) * 255).astype(np.uint8))
    cv2.imwrite(base + "_tumour.png", ((s == 2) * 255).astype(np.uint8))
print("saved", len(chosen), "slices to", OUT)

saved 22 slices to ../data/slices/evangeline


**Tumour-free slices (same patient).** To test "healthy" at the level of a whole slice (does the pipeline raise a false alarm on a slice with no tumour?), a few liver slices **without** tumour are saved in a separate folder `data/slices/<name>_healthy/`, so the Phase 1-3 test sets stay unchanged. Rules: liver at least 3000 pixels (no tiny pole slices), at least 3 slices away from any tumour slice, up to 6 slices spread evenly.

In [5]:
HEALTHY_OUT = f"../data/slices/{NAME}_healthy"
MIN_LIVER_PX, GAP, N_HEALTHY = 3000, 3, 6
liver_ok = [i for i in range(seg.shape[2]) if (seg[:, :, i] >= 1).sum() >= MIN_LIVER_PX]
free = [i for i in liver_ok if all(abs(i - t) >= GAP for t in tumour_slices)]          # no tumour within 3 slices
picks = np.unique(np.linspace(0, len(free) - 1, min(N_HEALTHY, len(free))).round().astype(int))
healthy_slices = [free[j] for j in picks]
os.makedirs(HEALTHY_OUT, exist_ok=True)
for i in healthy_slices:
    ct = np.rot90(vol[:, :, i])
    s = np.rot90(seg[:, :, i])
    base = f"{HEALTHY_OUT}/p{PATIENT}_s{i}"
    np.save(base + "_ct.npy", ct.astype(np.float32))
    cv2.imwrite(base + "_liver.png", ((s >= 1) * 255).astype(np.uint8))
    cv2.imwrite(base + "_tumour.png", ((s == 2) * 255).astype(np.uint8))      # all black: no tumour
print(len(free), "usable tumour-free liver slices; saved", healthy_slices, "to", HEALTHY_OUT)

76 usable tumour-free liver slices; saved [80, 95, 110, 125, 140, 155] to ../data/slices/evangeline_healthy
